# RNA-seq Mini-Capstone — DEK Overexpression in Adult Neural Stem Cells

**Dataset:** GEO accession **GSE200000**

**Public source:**  
- OmicsDI: https://www.omicsdi.org/dataset/geo/GSE200000  
- GEO sample metadata example: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM6001739

This notebook walks through an end-to-end analysis of processed RNA-seq expression data from adult mouse neural stem cells.

The biological comparison is:

> **How does conditional DEK overexpression alter the transcriptional state of adult neural stem cells?**

According to the GEO metadata, primary neural stem cells were isolated from the subventricular zone of adult mouse brain. Control cells were treated with AAV9-GFAP-GFP, while DEK conditional overexpression was induced with AAV9-GFAP-DEK-GFP. The supplied processed files contain three control samples and three DEK-overexpression samples.

The public GEO record describes the processed values as **RPKM**, not raw sequencing counts.

That distinction determines what analyses are defensible.

## Workflow

1. Load the two processed GEO files.
2. Clean column names and verify gene annotations.
3. Construct one expression matrix and sample metadata table.
4. Identify what RPKM values can and cannot tell us.
5. Inspect zeros and low-expression genes.
6. Filter weakly measured genes.
7. Apply a log2 transformation for exploratory analysis.
8. Compare sample-wide expression distributions.
9. Examine sample correlations.
10. Perform PCA and identify unusual samples.
11. Examine mitochondrial-expression burden as a QC clue.
12. Verify that the intended DEK perturbation is visible.
13. Estimate differential expression using a transparent teaching model.
14. Correct for multiple testing.
15. Make volcano, MA, and heatmap figures.
16. Examine biological candidate genes.
17. Perform leave-one-sample-out sensitivity analysis.
18. Use an exact permutation analysis to expose the consequences of \(n=3\) per group.
19. Summarize what conclusions are supported—and what would require raw counts or additional replication.

## Important limitation

This notebook is designed to teach statistical reasoning from the processed files that were supplied.

For a production RNA-seq differential-expression analysis, the preferred starting point would usually be **raw gene counts** analyzed with a count-aware method such as DESeq2, edgeR, or limma-voom.

## 1. Load the original processed GEO files

Place these two files in the same directory as the notebook:

- `GSE200000_Control.txt.gz`
- `GSE200000_DEK_cOE.txt.gz`

The loader also supports Google Colab upload if the files are not found locally.


In [ ]:
from pathlib import Path
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import gaussian_kde
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)

CONTROL_NAME = "GSE200000_Control.txt.gz"
DEK_NAME = "GSE200000_DEK_cOE.txt.gz"

control_path = Path(CONTROL_NAME)
dek_path = Path(DEK_NAME)

if not (control_path.exists() and dek_path.exists()):
    try:
        from google.colab import files

        print("Upload both GSE200000 .txt.gz files.")
        uploaded = files.upload()

        control_path = Path(CONTROL_NAME)
        dek_path = Path(DEK_NAME)

    except ImportError:
        raise FileNotFoundError(
            "Could not find both GSE200000 .txt.gz files "
            "in the current directory."
        )

print("Control file:", control_path)
print("DEK cOE file:", dek_path)


## 2. Inspect the tables before analyzing them

One subtle problem in these files is that some header fields contain leading whitespace.

That is a realistic data-cleaning issue: a column that visually appears to be called `gene_id` may actually contain spaces before the name.

We will therefore strip whitespace from every column header before doing anything else.


In [ ]:
control_raw = pd.read_csv(
    control_path,
    sep="\t",
    compression="gzip"
)

dek_raw = pd.read_csv(
    dek_path,
    sep="\t",
    compression="gzip"
)

print("Control shape:", control_raw.shape)
print("DEK cOE shape:", dek_raw.shape)

print("\nOriginal control columns:")
print(list(control_raw.columns))

print("\nOriginal DEK columns:")
print(list(dek_raw.columns))


In [ ]:
control = control_raw.copy()
dek = dek_raw.copy()

control.columns = [
    str(column).strip()
    for column in control.columns
]

dek.columns = [
    str(column).strip()
    for column in dek.columns
]

print("Cleaned control columns:")
print(list(control.columns))


## 3. Verify that the two files describe the same genes

Before joining two expression files, check:

- same number of rows;
- unique gene IDs;
- identical gene sets;
- compatible annotation.

The expression files contain extensive annotation, including:

- Ensembl gene ID;
- gene symbol;
- chromosome;
- coordinates;
- strand;
- gene length;
- gene biotype;
- description;
- transcription-factor family.


In [ ]:
control_samples = [
    "Control 1",
    "Control 2",
    "Control 3"
]

dek_samples = [
    "DEK cOE 1",
    "DEK cOE 2",
    "DEK cOE 3"
]

annotation_columns = [
    "gene_id",
    "gene_name",
    "gene_chr",
    "gene_start",
    "gene_end",
    "gene_strand",
    "gene_length",
    "gene_biotype",
    "gene_description",
    "tf_family"
]

print(
    "Unique control gene IDs:",
    control["gene_id"].nunique()
)

print(
    "Unique DEK gene IDs:",
    dek["gene_id"].nunique()
)

same_gene_set = (
    set(control["gene_id"])
    ==
    set(dek["gene_id"])
)

print(
    "Same gene set:",
    same_gene_set
)


In [ ]:
annotation_check = (
    control[annotation_columns]
    .merge(
        dek[annotation_columns],
        on="gene_id",
        suffixes=("_control", "_dek")
    )
)

annotation_differences = {}

for column in annotation_columns[1:]:

    left = annotation_check[
        column + "_control"
    ].astype(str)

    right = annotation_check[
        column + "_dek"
    ].astype(str)

    annotation_differences[column] = int(
        (left != right).sum()
    )

pd.Series(
    annotation_differences,
    name="rows_with_annotation_difference"
)


If the annotation is identical, we can keep one copy of it and join the six sample-expression columns by `gene_id`.


In [ ]:
expression_table = (
    control[
        annotation_columns
        + control_samples
    ]
    .merge(
        dek[
            ["gene_id"]
            + dek_samples
        ],
        on="gene_id",
        how="inner"
    )
)

sample_columns = (
    control_samples
    + dek_samples
)

sample_metadata = pd.DataFrame({
    "sample": sample_columns,
    "group": (
        ["Control"] * 3
        +
        ["DEK cOE"] * 3
    )
})

print(
    "Combined table:",
    expression_table.shape
)

sample_metadata


# 4. What are these expression values?

The GEO sample metadata describes the processed files as containing **RPKM** values.

RPKM stands for:

> **Reads Per Kilobase of transcript per Million mapped reads**

Conceptually,

$$
RPKM
=
\frac{\text{reads assigned to a gene}}
{\text{gene length in kb} \times \text{library size in millions}}.
$$

This attempts to account for:

- sequencing depth;
- gene length.

However, RPKM values are **not raw counts**.

That matters because count-based RNA-seq methods such as DESeq2 and edgeR model properties of integer sequencing counts, including the relationship between the mean and variance.

We should not feed RPKM values into those models and pretend they are counts.

For this capstone, we will use the processed RPKM values for:

- exploratory visualization;
- PCA;
- effect-size estimation;
- a transparent gene-wise comparison on the log scale;
- sensitivity analyses.

The statistical limitations will remain visible throughout the notebook.


## 5. Inspect the raw RPKM distribution

RNA-seq expression distributions are typically extremely right-skewed:

- many genes have zero or near-zero expression;
- a relatively small number of genes are expressed at very high levels.

Let's quantify that before transforming anything.


In [ ]:
expression = (
    expression_table[
        sample_columns
    ]
    .astype(float)
)

expression.describe(
    percentiles=[
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)


In [ ]:
zero_summary = pd.DataFrame({
    "sample": sample_columns,
    "zero_genes": [
        int(
            (expression[sample] == 0).sum()
        )
        for sample in sample_columns
    ],
    "percent_zero": [
        100
        * (
            expression[sample] == 0
        ).mean()
        for sample in sample_columns
    ]
})

zero_summary


A gene with essentially no measurable expression cannot provide a stable estimate of a treatment effect.

Filtering low-expression genes before thousands of hypothesis tests can:

- remove uninformative features;
- reduce the multiple-testing burden;
- prevent enormous fold changes driven by tiny denominators.

The threshold must be defined **before** looking at which genes become significant.

For this teaching analysis, retain genes with:

> **RPKM ≥ 1 in at least 3 of the 6 samples.**

This is a transparent heuristic for the processed data, not a universal RNA-seq rule.


In [ ]:
expression_filter = (
    (expression >= 1)
    .sum(axis=1)
    >= 3
)

filtered = (
    expression_table
    .loc[expression_filter]
    .reset_index(drop=True)
)

print(
    "Genes before filtering:",
    len(expression_table)
)

print(
    "Genes after filtering:",
    len(filtered)
)

print(
    "Genes removed:",
    len(expression_table)
    - len(filtered)
)


## 6. Log-transform the expression values

The raw RPKM scale spans orders of magnitude.

For visualization and simple continuous-data modeling, use:

$$
x' = \log_2(RPKM + 1).
$$

The `+1` allows zero-expression values to be transformed.

On this scale:

- a one-unit difference corresponds approximately to a two-fold difference when expression is not near zero;
- extreme high-expression genes have less leverage;
- sample distributions become easier to compare.


In [ ]:
log_expression = np.log2(
    filtered[
        sample_columns
    ].astype(float)
    + 1
)

log_expression.describe()


In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 5)
)

positions = np.arange(
    len(sample_columns)
)

ax.boxplot(
    [
        log_expression[
            sample
        ].to_numpy()
        for sample in sample_columns
    ],
    positions=positions,
    widths=0.6,
    showfliers=False
)

ax.set_xticks(
    positions
)

ax.set_xticklabels(
    sample_columns,
    rotation=45,
    ha="right"
)

ax.set_ylabel(
    "log2(RPKM + 1)"
)

ax.set_title(
    "Genome-wide expression distributions"
)

plt.tight_layout()
plt.show()


# 7. Sample-level quality control

Before testing individual genes, ask whether the **samples** behave sensibly.

Useful questions include:

- Do biological replicates correlate with one another?
- Do treatment groups separate globally?
- Is one sample unusual relative to the others?

These questions should be answered before looking at a volcano plot.


## 7.1 Pairwise sample correlations

Correlation asks whether genes that are relatively high in one sample also tend to be relatively high in another.

Biological replicates should usually be more similar to each other than unrelated samples, although real biological heterogeneity can complicate that expectation.


In [ ]:
sample_correlation = (
    log_expression
    .corr()
)

fig, ax = plt.subplots(
    figsize=(7, 6)
)

im = ax.imshow(
    sample_correlation,
    vmin=0.80,
    vmax=1.00,
    cmap="viridis"
)

ax.set_xticks(
    np.arange(
        len(sample_columns)
    )
)

ax.set_yticks(
    np.arange(
        len(sample_columns)
    )
)

ax.set_xticklabels(
    sample_columns,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(
    sample_columns
)

for i in range(
    len(sample_columns)
):
    for j in range(
        len(sample_columns)
    ):
        ax.text(
            j,
            i,
            f"{sample_correlation.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.colorbar(
    im,
    ax=ax,
    label="Pearson correlation"
)

ax.set_title(
    "Sample-to-sample correlation"
)

plt.tight_layout()
plt.show()


## 7.2 Principal component analysis

PCA reduces thousands of gene-expression measurements to a few axes that capture the largest patterns of variation.

Each point is now an **entire RNA-seq sample**, not an individual gene.

If treatment is a major source of variation, samples from the two groups may separate along one of the leading principal components.

An unusual sample should be **investigated**, not automatically deleted.


In [ ]:
# PCA centers each gene across samples.
X_pca = (
    log_expression
    .T
)

pca = PCA(
    n_components=2
)

pca_coordinates = pca.fit_transform(
    X_pca
)

pca_df = pd.DataFrame({
    "sample": sample_columns,
    "group": sample_metadata[
        "group"
    ],
    "PC1": pca_coordinates[
        :,
        0
    ],
    "PC2": pca_coordinates[
        :,
        1
    ]
})

pca_df


In [ ]:
fig, ax = plt.subplots(
    figsize=(7, 6)
)

for group_name, group_df in pca_df.groupby(
    "group"
):

    ax.scatter(
        group_df["PC1"],
        group_df["PC2"],
        s=100,
        label=group_name
    )

    for _, row in group_df.iterrows():

        ax.text(
            row["PC1"] + 1,
            row["PC2"] + 1,
            row["sample"],
            fontsize=9
        )

ax.set_xlabel(
    (
        "PC1 "
        f"({100 * pca.explained_variance_ratio_[0]:.1f}% variance)"
    )
)

ax.set_ylabel(
    (
        "PC2 "
        f"({100 * pca.explained_variance_ratio_[1]:.1f}% variance)"
    )
)

ax.set_title(
    "PCA of filtered log-expression data"
)

ax.legend()

plt.show()


### Interpretation prompt

You should notice two things:

1. the DEK cOE samples separate from the controls along a major axis;
2. `Control 3` differs substantially from `Control 1` and `Control 2`.

That does **not** prove Control 3 is technically bad.

Possible explanations include:

- genuine biological heterogeneity;
- RNA quality differences;
- library preparation differences;
- sequencing-depth/composition effects;
- differences in cell state or cell composition.

Before removing a sample, we would want experimental or sequencing-QC evidence.


## 7.3 Mitochondrial-expression burden

One QC clue is the relative contribution of mitochondrial genes.

Because we have RPKM rather than raw reads, this is **not the same as the usual percent mitochondrial reads metric** used in raw-count QC.

Still, the fraction of the summed RPKM contributed by mitochondrial genes can be used as an exploratory composition diagnostic.


In [ ]:
full_expression = (
    expression_table[
        sample_columns
    ]
    .astype(float)
)

is_mitochondrial = (
    expression_table[
        "gene_chr"
    ].astype(str).eq("MT")
    |
    expression_table[
        "gene_name"
    ].astype(str).str.startswith(
        "mt-"
    )
)

mitochondrial_summary = pd.DataFrame({
    "sample": sample_columns,
    "total_RPKM_sum": [
        full_expression[
            sample
        ].sum()
        for sample in sample_columns
    ],
    "mitochondrial_RPKM_sum": [
        full_expression.loc[
            is_mitochondrial,
            sample
        ].sum()
        for sample in sample_columns
    ]
})

mitochondrial_summary[
    "mitochondrial_fraction_of_RPKM_sum"
] = (
    mitochondrial_summary[
        "mitochondrial_RPKM_sum"
    ]
    /
    mitochondrial_summary[
        "total_RPKM_sum"
    ]
)

mitochondrial_summary


In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 4.5)
)

ax.bar(
    mitochondrial_summary[
        "sample"
    ],
    100
    * mitochondrial_summary[
        "mitochondrial_fraction_of_RPKM_sum"
    ]
)

ax.set_ylabel(
    "Mitochondrial fraction of summed RPKM (%)"
)

ax.set_title(
    "Exploratory mitochondrial-expression composition"
)

ax.tick_params(
    axis="x",
    rotation=45
)

plt.tight_layout()
plt.show()


Control 3 has a noticeably larger mitochondrial contribution than Controls 1–2.

That observation is a **QC flag**, not an exclusion rule.

The primary analysis below retains all six samples. Later, leave-one-sample-out sensitivity analysis will show how strongly the estimated treatment effects depend on individual samples.


# 8. Verify that the intended perturbation is visible

Before performing genome-wide inference, check the manipulated gene itself.

If the experiment is labeled `DEK cOE`, we should ask:

> Is `Dek` actually more highly expressed in the overexpression samples?

This is a useful positive-control check.


In [ ]:
dek_gene = filtered.loc[
    filtered[
        "gene_name"
    ].astype(str).str.lower()
    == "dek"
]

dek_gene[
    [
        "gene_id",
        "gene_name"
    ]
    + sample_columns
]


In [ ]:
dek_values = (
    dek_gene[
        sample_columns
    ]
    .iloc[0]
    .astype(float)
)

fig, ax = plt.subplots(
    figsize=(7, 4.5)
)

x = np.arange(
    len(sample_columns)
)

ax.bar(
    x,
    dek_values.values
)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    sample_columns,
    rotation=45,
    ha="right"
)

ax.set_ylabel(
    "Dek expression (RPKM)"
)

ax.set_title(
    "Positive-control check: DEK overexpression"
)

plt.tight_layout()
plt.show()


# 9. Experimental unit and sample size

There are:

- **3 Control samples**
- **3 DEK cOE samples**

The experimental unit is the biological sample represented by each RNA-seq library.

Genes are **not replicates**.

The presence of more than 10,000 measured genes does not turn \(n=3\) into a large experiment.

This distinction becomes critical for uncertainty estimation.


# 10. A transparent differential-expression teaching model

Because the supplied values are processed RPKM rather than raw counts, we will use a simple continuous-data comparison on

$$
\log_2(RPKM + 1).
$$

For each gene, estimate:

$$
\widehat{\log_2 FC}
=
\overline{x}_{DEK}
-
\overline{x}_{Control}.
$$

We then use a **Welch two-sample t-test**, which allows the two groups to have different variances.

This is deliberately transparent, but it has important limitations:

- only three observations per group;
- gene-wise variance estimates are unstable;
- the method does not model RNA-seq counts;
- it does not provide the empirical-Bayes variance moderation used by tools such as limma.

Treat this as a teaching analysis of processed values, not as a replacement for a production raw-count pipeline.


In [ ]:
control_log = (
    log_expression[
        control_samples
    ]
)

dek_log = (
    log_expression[
        dek_samples
    ]
)

mean_control = control_log.mean(
    axis=1
)

mean_dek = dek_log.mean(
    axis=1
)

log2fc = (
    mean_dek
    -
    mean_control
)

t_statistic, p_value = stats.ttest_ind(
    dek_log.to_numpy(),
    control_log.to_numpy(),
    axis=1,
    equal_var=False,
    nan_policy="omit"
)

de_results = filtered[
    [
        "gene_id",
        "gene_name",
        "gene_biotype",
        "tf_family"
    ]
].copy()

de_results[
    "mean_control_log2"
] = mean_control.to_numpy()

de_results[
    "mean_DEK_log2"
] = mean_dek.to_numpy()

de_results[
    "log2FC"
] = log2fc.to_numpy()

de_results[
    "t_statistic"
] = t_statistic

de_results[
    "p_value"
] = p_value

de_results[
    "FDR"
] = multipletests(
    de_results[
        "p_value"
    ],
    method="fdr_bh"
)[1]

de_results[
    "significant"
] = (
    (
        de_results[
            "FDR"
        ] < 0.05
    )
    &
    (
        np.abs(
            de_results[
                "log2FC"
            ]
        ) >= 1
    )
)

print(
    "Genes tested:",
    len(
        de_results
    )
)

print(
    "FDR < 0.05:",
    int(
        (
            de_results[
                "FDR"
            ] < 0.05
        ).sum()
    )
)

print(
    "FDR < 0.05 and |log2FC| >= 1:",
    int(
        de_results[
            "significant"
        ].sum()
    )
)

de_results.sort_values(
    "FDR"
).head(20)


## 10.1 Check DEK in the differential-expression table

A useful sanity check is whether the manipulated gene is recovered as a large positive effect.


In [ ]:
de_results.loc[
    de_results[
        "gene_name"
    ].astype(str).str.lower()
    == "dek"
]


# 11. Volcano plot

The volcano plot combines:

- effect size on the x-axis;
- FDR-adjusted statistical evidence on the y-axis.

The plot is useful for orientation, but it can create false confidence when sample size is very small.

A point high on the plot means the **parametric model** estimates strong evidence—not that the experimental design suddenly contains many biological replicates.


In [ ]:
volcano = de_results.copy()

volcano[
    "minus_log10_FDR"
] = -np.log10(
    volcano[
        "FDR"
    ].clip(
        lower=1e-300
    )
)

fig, ax = plt.subplots(
    figsize=(8.5, 6)
)

non_sig = ~volcano[
    "significant"
]

sig = volcano[
    "significant"
]

ax.scatter(
    volcano.loc[
        non_sig,
        "log2FC"
    ],
    volcano.loc[
        non_sig,
        "minus_log10_FDR"
    ],
    alpha=0.35,
    s=18,
    label="Not selected"
)

ax.scatter(
    volcano.loc[
        sig,
        "log2FC"
    ],
    volcano.loc[
        sig,
        "minus_log10_FDR"
    ],
    alpha=0.75,
    s=28,
    label="FDR < 0.05 and |log2FC| ≥ 1"
)

ax.axvline(
    -1,
    linestyle="--",
    linewidth=1
)

ax.axvline(
    1,
    linestyle="--",
    linewidth=1
)

ax.axhline(
    -np.log10(
        0.05
    ),
    linestyle="--",
    linewidth=1
)

# Label DEK.
dek_volcano = volcano.loc[
    volcano[
        "gene_name"
    ] == "Dek"
]

if len(
    dek_volcano
) == 1:

    x_dek = float(
        dek_volcano[
            "log2FC"
        ].iloc[0]
    )

    y_dek = float(
        dek_volcano[
            "minus_log10_FDR"
        ].iloc[0]
    )

    ax.annotate(
        "Dek",
        (
            x_dek,
            y_dek
        ),
        xytext=(
            x_dek + 0.3,
            y_dek + 0.3
        ),
        arrowprops={
            "arrowstyle": "->"
        }
    )

ax.set_xlabel(
    "log2 fold change"
)

ax.set_ylabel(
    "-log10(FDR)"
)

ax.set_title(
    "DEK cOE versus Control"
)

ax.legend()

plt.show()


# 12. MA-style plot

A volcano plot emphasizes statistical evidence.

An MA-style plot emphasizes the relationship between:

- average expression;
- estimated effect size.

This can reveal whether enormous fold changes are concentrated among weakly expressed genes.


In [ ]:
de_results[
    "average_log_expression"
] = (
    de_results[
        "mean_control_log2"
    ]
    +
    de_results[
        "mean_DEK_log2"
    ]
) / 2

fig, ax = plt.subplots(
    figsize=(8.5, 5.5)
)

ax.scatter(
    de_results[
        "average_log_expression"
    ],
    de_results[
        "log2FC"
    ],
    alpha=0.35,
    s=18
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.axhline(
    1,
    linestyle=":",
    linewidth=1
)

ax.axhline(
    -1,
    linestyle=":",
    linewidth=1
)

ax.set_xlabel(
    "Average log2(RPKM + 1)"
)

ax.set_ylabel(
    "log2 fold change"
)

ax.set_title(
    "Effect size versus expression level"
)

plt.show()


# 13. Heatmap of the strongest signals

A differential-expression table compresses each gene into a few statistics.

A heatmap returns to the **six original samples**.

This is important because a very small p-value can still be driven by a particular sample pattern.

We will display the 25 genes with the smallest FDR values and standardize each gene across samples:

$$
z_{gs}
=
\frac{x_{gs}-\bar{x}_g}
{s_g}.
$$

This emphasizes relative sample patterns rather than absolute expression magnitude.


In [ ]:
top_genes = (
    de_results
    .sort_values(
        "FDR"
    )
    .head(25)[
        "gene_id"
    ]
)

top_mask = filtered[
    "gene_id"
].isin(
    top_genes
)

heat = (
    log_expression.loc[
        top_mask
    ]
    .copy()
)

heat.index = (
    filtered.loc[
        top_mask,
        "gene_name"
    ]
    .fillna(
        filtered.loc[
            top_mask,
            "gene_id"
        ]
    )
    .astype(str)
)

heat = heat.loc[
    [
        gene_name
        for gene_name in (
            de_results
            .sort_values(
                "FDR"
            )
            .head(25)[
                "gene_name"
            ]
            .astype(str)
        )
        if gene_name in heat.index
    ]
]

heat_z = (
    heat
    .sub(
        heat.mean(
            axis=1
        ),
        axis=0
    )
    .div(
        heat.std(
            axis=1
        ).replace(
            0,
            np.nan
        ),
        axis=0
    )
)

fig, ax = plt.subplots(
    figsize=(9, 9)
)

im = ax.imshow(
    heat_z,
    aspect="auto",
    cmap="coolwarm",
    vmin=-2.5,
    vmax=2.5
)

ax.set_xticks(
    np.arange(
        len(sample_columns)
    )
)

ax.set_xticklabels(
    sample_columns,
    rotation=45,
    ha="right"
)

ax.set_yticks(
    np.arange(
        len(heat_z.index)
    )
)

ax.set_yticklabels(
    heat_z.index
)

ax.set_title(
    "Top differential-expression signals"
)

plt.colorbar(
    im,
    ax=ax,
    label="Gene-wise z-score"
)

plt.tight_layout()
plt.show()


# 14. A biologically motivated candidate-gene panel

The public study description connects DEK overexpression with neural-stem-cell quiescence exit and downstream Notch signaling.

Rather than searching only for the smallest p-values, inspect a small hypothesis-driven panel containing:

- `Dek`;
- Notch-pathway genes;
- neural stem-cell / progenitor markers;
- proliferation-associated genes.

Candidate-gene inspection is **not a replacement** for genome-wide correction. It asks a different question: do genes motivated by the biological hypothesis show coherent expression patterns?


In [ ]:
candidate_genes = [
    "Dek",
    "Notch1",
    "Notch2",
    "Hes1",
    "Hes5",
    "Rbpj",
    "Ascl1",
    "Sox2",
    "Mki67",
    "Pcna",
    "Ccnd1",
    "Dcx",
    "Nes",
    "Gfap"
]

candidate_results = (
    de_results.loc[
        de_results[
            "gene_name"
        ].isin(
            candidate_genes
        ),
        [
            "gene_name",
            "mean_control_log2",
            "mean_DEK_log2",
            "log2FC",
            "p_value",
            "FDR"
        ]
    ]
    .sort_values(
        "log2FC",
        ascending=False
    )
)

candidate_results


In [ ]:
candidate_order = [
    gene
    for gene in candidate_genes
    if gene in set(
        filtered[
            "gene_name"
        ].astype(str)
    )
]

candidate_mask = filtered[
    "gene_name"
].isin(
    candidate_order
)

candidate_heat = (
    log_expression.loc[
        candidate_mask
    ]
    .copy()
)

candidate_heat.index = filtered.loc[
    candidate_mask,
    "gene_name"
].astype(str)

candidate_heat = candidate_heat.reindex(
    candidate_order
)

candidate_z = (
    candidate_heat
    .sub(
        candidate_heat.mean(
            axis=1
        ),
        axis=0
    )
    .div(
        candidate_heat.std(
            axis=1
        ).replace(
            0,
            np.nan
        ),
        axis=0
    )
)

fig, ax = plt.subplots(
    figsize=(8.5, 6.5)
)

im = ax.imshow(
    candidate_z,
    aspect="auto",
    cmap="coolwarm",
    vmin=-2.5,
    vmax=2.5
)

ax.set_xticks(
    np.arange(
        len(sample_columns)
    )
)

ax.set_xticklabels(
    sample_columns,
    rotation=45,
    ha="right"
)

ax.set_yticks(
    np.arange(
        len(candidate_z.index)
    )
)

ax.set_yticklabels(
    candidate_z.index
)

ax.set_title(
    "Hypothesis-driven neural stem-cell gene panel"
)

plt.colorbar(
    im,
    ax=ax,
    label="Gene-wise z-score"
)

plt.tight_layout()
plt.show()


# 15. Small-\(n\) sensitivity: leave one sample out

With only three samples per group, every individual sample carries substantial weight.

A useful robustness check is to recalculate the log2 fold change after leaving out each sample one at a time.

For each gene we can ask:

> Does the direction and approximate magnitude of the estimated effect survive the loss of one sample?

This is **not** a substitute for more biological replication.

It is a way to expose fragility.


In [ ]:
full_log2fc = (
    log_expression[
        dek_samples
    ].mean(
        axis=1
    )
    -
    log_expression[
        control_samples
    ].mean(
        axis=1
    )
)

leave_one_out_effects = {}

for omitted_sample in sample_columns:

    remaining_controls = [
        sample
        for sample in control_samples
        if sample != omitted_sample
    ]

    remaining_dek = [
        sample
        for sample in dek_samples
        if sample != omitted_sample
    ]

    loo_effect = (
        log_expression[
            remaining_dek
        ].mean(
            axis=1
        )
        -
        log_expression[
            remaining_controls
        ].mean(
            axis=1
        )
    )

    leave_one_out_effects[
        omitted_sample
    ] = loo_effect

loo_effects = pd.DataFrame(
    leave_one_out_effects
)

loo_correlations = {
    sample: np.corrcoef(
        full_log2fc,
        loo_effects[
            sample
        ]
    )[0, 1]
    for sample in sample_columns
}

pd.Series(
    loo_correlations,
    name="correlation_with_full_log2FC"
)


In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 4.5)
)

ax.bar(
    list(
        loo_correlations.keys()
    ),
    list(
        loo_correlations.values()
    )
)

ax.set_ylim(
    0,
    1
)

ax.set_ylabel(
    "Correlation with full-data log2FC"
)

ax.set_title(
    "Leave-one-sample-out effect-size stability"
)

ax.tick_params(
    axis="x",
    rotation=45
)

plt.tight_layout()
plt.show()


## 15.1 Inspect DEK itself under leave-one-out analysis


In [ ]:
dek_index = filtered.index[
    filtered[
        "gene_name"
    ] == "Dek"
][0]

dek_stability = pd.DataFrame({
    "analysis": (
        ["All samples"]
        +
        [
            f"Leave out {sample}"
            for sample in sample_columns
        ]
    ),
    "Dek_log2FC": (
        [
            full_log2fc.loc[
                dek_index
            ]
        ]
        +
        [
            loo_effects.loc[
                dek_index,
                sample
            ]
            for sample in sample_columns
        ]
    )
})

dek_stability


# 16. Exact permutation reality check

This dataset gives us a powerful lesson about sample size.

There are six samples total and we assign three to each group.

The number of possible 3-versus-3 group assignments is

$$
\binom{6}{3}
=
20.
$$

That means an exact randomization/permutation analysis has only 20 possible label configurations.

For a two-sided test, even an extremely dramatic separation can have very coarse p-value resolution.

Let's demonstrate this for `Dek`.

The observed statistic will be the difference in mean log2 expression:

$$
T
=
\bar{x}_{DEK}
-
\bar{x}_{Control}.
$$


In [ ]:
dek_log_values = (
    log_expression.loc[
        dek_index,
        sample_columns
    ]
    .to_numpy(
        dtype=float
    )
)

observed_dek_difference = (
    dek_log_values[
        3:
    ].mean()
    -
    dek_log_values[
        :3
    ].mean()
)

permuted_differences = []

sample_indices = np.arange(
    6
)

for treatment_indices in itertools.combinations(
    sample_indices,
    3
):

    treatment_indices = np.array(
        treatment_indices
    )

    control_indices_perm = np.array([
        index
        for index in sample_indices
        if index not in treatment_indices
    ])

    permuted_difference = (
        dek_log_values[
            treatment_indices
        ].mean()
        -
        dek_log_values[
            control_indices_perm
        ].mean()
    )

    permuted_differences.append(
        permuted_difference
    )

permuted_differences = np.asarray(
    permuted_differences
)

exact_two_sided_p = np.mean(
    np.abs(
        permuted_differences
    )
    >=
    abs(
        observed_dek_difference
    )
    - 1e-12
)

print(
    "Observed Dek log2 difference:",
    round(
        observed_dek_difference,
        3
    )
)

print(
    "Number of possible assignments:",
    len(
        permuted_differences
    )
)

print(
    "Exact two-sided permutation p-value:",
    exact_two_sided_p
)


In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 4.5)
)

ax.hist(
    permuted_differences,
    bins=12,
    edgecolor="black"
)

ax.axvline(
    observed_dek_difference,
    linewidth=2.5,
    label="Observed DEK cOE − Control"
)

ax.axvline(
    -observed_dek_difference,
    linewidth=1.5,
    linestyle="--"
)

ax.set_xlabel(
    "Permuted difference in mean log2 expression"
)

ax.set_ylabel(
    "Number of label assignments"
)

ax.set_title(
    "Exact permutation distribution for Dek"
)

ax.legend()

plt.show()


## Why can the t-test and permutation test look so different?

The Welch t-test uses a parametric model.

It assumes that the six observations are draws from underlying continuous distributions and uses that model to estimate tail probabilities far smaller than the number of observed label arrangements.

The exact permutation test asks a stricter design-based question:

> If the six observed samples were exchangeable under the null, how unusual is this group assignment among all possible 3-versus-3 assignments?

With only 20 assignments, the evidence is necessarily coarse.

For `Dek`, the biological effect is enormous and the overexpression manipulation is obvious, yet the exact two-sided permutation p-value cannot become extremely small.

This is not a contradiction.

It illustrates that:

> **effect size, model-based p-values, and information supplied by the experimental design are different things.**


# 17. Why we should not automatically delete Control 3

PCA and mitochondrial composition both identify Control 3 as unusual relative to Controls 1–2.

That is useful information.

It is not enough information to remove the sample.

A defensible exclusion would require evidence such as:

- failed RNA-quality metrics;
- documented library-preparation failure;
- sample mislabeling;
- extreme sequencing-quality problems;
- a prespecified QC threshold.

Deleting a sample because it makes the volcano plot look better would be circular.

The primary analysis therefore keeps Control 3, while the leave-one-out analysis shows students exactly how much individual samples influence the results.


# 18. What can we conclude?

From these processed files we can reasonably conclude that:

1. the DEK-overexpression manipulation is strongly visible in `Dek` expression;
2. genome-wide expression differs substantially between the two groups;
3. many genes have large estimated effects under the log-RPKM teaching analysis;
4. the sample-level structure is not perfectly homogeneous, particularly among the controls;
5. statistical certainty is constrained by only three biological samples per group.

We should be more cautious about claims such as:

- a precise genome-wide list of "true" differentially expressed genes;
- exact pathway significance based solely on these six processed samples;
- causal molecular mechanisms inferred from expression alone.

For a production reanalysis, the ideal next step would be to obtain the **raw gene-count matrix or FASTQ files** and use a count-aware RNA-seq workflow.


# 19. Methods-style summary

A concise description of this teaching analysis could read:

> Processed RPKM expression matrices for three control and three DEK conditional-overexpression neural stem-cell samples from GEO accession GSE200000 were imported and joined by Ensembl gene identifier. Genes with RPKM ≥ 1 in at least three samples were retained. Expression values were transformed as log2(RPKM + 1) for exploratory analysis. Sample similarity was evaluated using Pearson correlation and principal component analysis. Gene-level treatment effects were summarized as differences in mean log2 expression between DEK cOE and control samples, with Welch two-sample t-tests used as a transparent teaching approximation for the processed continuous measurements. Benjamini-Hochberg false-discovery-rate adjustment was applied across tested genes. Robustness was evaluated using leave-one-sample-out effect-size analysis and an exact permutation demonstration. Because the supplied values were processed RPKM rather than raw counts and the experiment contained three samples per group, differential-expression inference was interpreted cautiously.

For a manuscript, the statistical workflow should match the actual raw-data processing and the study's prespecified analysis plan.


# 20. Final capstone questions

## Data provenance

1. What do the two uploaded files contain?
2. Why did the column headers need cleaning?
3. How did we verify that the two files contained the same gene annotation?
4. What does GEO say the expression values represent?

## Preprocessing

5. Why are RPKM values different from raw RNA-seq counts?
6. Why did we filter low-expression genes?
7. Why did we use log2(RPKM + 1) for visualization and the teaching model?
8. How would the workflow differ if raw counts were available?

## Sample QC

9. What did the sample-correlation matrix show?
10. What did PCA show?
11. Why is Control 3 worth investigating?
12. Why is the mitochondrial RPKM fraction only an exploratory QC measure here?
13. What evidence would justify excluding Control 3?

## Differential expression

14. What does log2FC represent?
15. What does an FDR-adjusted p-value represent?
16. Why is \(n=3\) per group a major limitation despite measuring thousands of genes?
17. Why should the heatmap be inspected in addition to the volcano plot?
18. Why are very large effects among weakly expressed genes potentially unstable?

## Sensitivity and model assumptions

19. What does leave-one-sample-out analysis reveal?
20. What does it fail to solve?
21. Why are there only 20 possible 3-versus-3 label assignments?
22. Why can the Welch t-test produce a tiny p-value when the exact permutation p-value is coarse?
23. Which analysis relies more strongly on parametric distributional assumptions?

## Biological interpretation

24. Does the dataset confirm that DEK overexpression occurred?
25. Which candidate genes are consistent with altered neural-stem-cell state?
26. Why does expression change alone not establish molecular causality?
27. What additional experiment or data would strengthen the biological conclusions?


# Key idea

A transcriptomic analysis is not:

> load expression matrix → run 13,000 tests → report the smallest p-values.

It is:

> **understand where the values came from → identify the experimental unit → inspect sample quality → define filtering and transformation rules → visualize the sample structure → estimate biologically meaningful effects → account for multiplicity → test robustness to individual samples → distinguish model-based certainty from design-based information → communicate the limits of the experiment.**
